RAG system  -data ingestion pipeline

In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
##document loading

loader=DirectoryLoader(
    "../data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

loaded_documents=loader.load()

page_texts="\n".join(doc.page_content for doc in loaded_documents)

print(f"loaded documents: {len(loaded_documents)}")
# print(f"page texts: {page_texts}")

loaded documents: 14


In [10]:
import re
from langchain_core.documents import Document


In [ ]:
##document splitting and chunking into a structured doc


from pydoc import text
from re import IGNORECASE



chunked_program=re.split(r"Degree programme",page_texts)
structured_docs=[]

for prog in chunked_program:
    chunked_program_match=re.search(r"^(.*?)\n",prog)
    program_name=chunked_program_match.group(1).strip() if chunked_program_match else "Unknown Program"

    Years=re.split(r"Year (one|two|three|four)",prog, flags=IGNORECASE)
    for i in range(1,len(Years),2):
        year=Years[i].strip()
        year_content=Years[i+1].strip()


        Semesters=re.split(r"Semester (one|two)",year_content)
        for j in range(1,len(Semesters),2):
            semester=Semesters[j].strip()
            semester_content=Semesters[j+1].strip()

            structured_docs.append({
                "text": semester_content,
                "metadata": {
                    "Degree Program": program_name,
                    "Year": year,
                    "Semester": semester
                }
            })


        def cleaned_course(text):
            lines=text.split("\n")
            course=[]

            for line in lines:
                parts=line.strip().split()
                if len(parts)>3:
                    course_name=" ".join(parts[2:-2]).strip()
                    course.append(course_name)
            return "course\n"+ "\n".join(f"- {c}" for c in course)
        
        full_chunked_docs=[]
        for doc in structured_docs:
            cleaned_text=cleaned_course(doc["text"])
            full_chunked_docs.append(Document(
                page_content=cleaned_text,
                metadata=doc["metadata"]
            ))

        ##print(f"structured documents: {len(full_chunked_docs)}")


           





In [15]:
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_chroma import Chroma
import os

In [ ]:
##vector storage and embedding

from openai import embeddings


embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

root_path=os.path.dirname(os.getcwd())
persist_directory=os.path.join(root_path,"chromaDB")

vector_stored=Chroma.from_documents(
    documents=full_chunked_docs,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="syllabus_pdfs"

)

print(f"vectored stored successfully\n and stored here: {persist_directory}")

vectored stored successfully
 and stored here: d:\flow\openai_demo\chromaDB


retrieval pipeline , from vector store.

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   
from langchain_core.runnables import RunnablePassthrough

In [24]:
from langchain_core.runnables import RunnableMap
from opentelemetry import context


retriever=vector_stored.as_retriever(search_kwargs={"k":10})

system_prompt= ( "You are an academic auditor. This document contains multiple different degree programs. "
    "When asked a question: "
    "1. Find the exact 'Degree Programme' header first (e.g., BSc in Computer Engineering). "
    "2. Only look at the tables following that specific header. "
    "3. If a course appears in a DIFFERENT program's table, do not attribute it to the requested program. "
    "4. If the course is not in the requested program, say 'This course is not offered in this specific program.' "
    "\n\nContext: {context}"
    )


prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}")
    ]
)

llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)


rag_chain=(
    RunnableMap({
        "context":lambda x: retriever.invoke(x["question"]),
        "question":lambda x: x["question"]


    }) 

    | prompt
    | llm   
    | StrOutputParser()
)


response=rag_chain.invoke({"question": "What are degree program offered "})

print(f"retriever response: {response}")

retriever response: The document contains the following degree program:

- Bachelor of Science in Computer Networks and Information Security
